In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.utils import save_image, make_grid
import os

batch_size = 128
z_dim = 100 # Size of noise vector
num_classes = 10
image_size = 28
channels = 1
epochs = 50
lr = 0.0002
beta1 = 0.5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs("cgan_generated", exist_ok=True)

In [16]:
transform = transforms.Compose([
    #transforms.Resize(image_size),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

train_loader = torch.utils.data.DataLoader(
    datasets.MNIST('.', train=True, download=True, transform=transform),
    batch_size=batch_size,
    shuffle=True
)

In [17]:
class Generator(nn.Module):
    def __init__(self, z_dim, num_classes, img_shape):
        super(Generator, self).__init__()
        self.label_emb = nn.Embedding(num_classes, num_classes)
        self.img_shape = img_shape
        input_dim = z_dim + num_classes
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(True),
            
            nn.Linear(256, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(True),
            
            nn.Linear(512, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(True),
            
            nn.Linear(1024, int(torch.prod(torch.tensor(img_shape)))),
            nn.Tanh()
        )
    
    def forward(self, z, labels):
        x = torch.cat([z, self.label_emb(labels)], dim=1)
        img = self.net(x)
        img = img.view(x.size(0), *self.img_shape)
        return img

In [18]:
class Discriminator(nn.Module):
    def __init__(self, num_classes, img_shape):
        super(Discriminator, self).__init__()
        self.label_emb = nn.Embedding(num_classes, num_classes)
        #self.img_shape = img_shape
        input_dim = int(torch.prod(torch.tensor(img_shape))) + num_classes
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Linear(256, 1),
            nn.Sigmoid()
        )
    
    def forward(self, img, labels):
        img_flat = img.view(img.size(0), -1)
        x = torch.cat([img_flat, self.label_emb(labels)], dim=1)
        return self.net(x)

In [19]:
img_shape = (channels, image_size, image_size)

generator = Generator(z_dim, num_classes, img_shape).to(device)
discriminator = Discriminator(num_classes, img_shape).to(device)

criterion = nn.BCELoss()

optimizer_G = optim.Adam(generator.parameters(), lr=lr, betas=(beta1, 0.999))
optimizer_D = optim.Adam(discriminator.parameters(), lr=lr, betas=(beta1, 0.999))

In [20]:
k = 3
p = 1

In [22]:
for epoch in range(1, epochs + 1):
    for i, (real_imgs, real_labels) in enumerate(train_loader):
        batch_size_curr = real_imgs.size(0)
        real_imgs = real_imgs.to(device)
        real_labels = real_labels.to(device)
        
        real = torch.ones(batch_size_curr, 1, device=device)
        fake = torch.zeros(batch_size_curr, 1, device=device)
        
        for _ in range(p):
            z = torch.randn(batch_size_curr, z_dim, device=device)
            fake_labels = torch.randint(0, num_classes, (batch_size_curr,), device=device)
            
            with torch.no_grad():
                fake_imgs = generator(z, fake_labels)
            
            real_validity = discriminator(real_imgs, real_labels)
            d_real_loss = criterion(real_validity, real)
            
            fake_validity = discriminator(fake_imgs.detach(), fake_labels)
            d_fake_loss = criterion(fake_validity, fake)
            
            d_loss = d_real_loss + d_fake_loss
            
            optimizer_D.zero_grad()
            d_loss.backward()
            optimizer_D.step()
        
        for _ in range(k):
            z = torch.randn(batch_size_curr, z_dim, device=device)
            fake_labels = torch.randint(0, num_classes, (batch_size_curr,), device=device)
            fake_imgs = generator(z, fake_labels)
            
            validity = discriminator(fake_imgs, fake_labels)
            g_loss = criterion(validity, real) # fool D: label as real
            
            optimizer_G.zero_grad()
            g_loss.backward()
            optimizer_G.step()
        
        if i % 200 == 0:
            print(f"[Epoch {epoch}/{epochs}] [Batch {i}/{len(train_loader)}] "
                  f"D Loss: {d_loss.item():.4f} | G Loss: {g_loss.item():.4f}")
    
    generator.eval()
    with torch.no_grad():
        z = torch.randn(10, z_dim, device=device)
        labels = torch.arange(0, 10, dtype=torch.long, device=device)
        samples = generator(z, labels)
        samples = samples * 0.5 + 0.5
        save_image(samples, f"cgan_generated/epoch_{epoch}.png", nrow=10)
    generator.train()

[Epoch 1/50] [Batch 0/469] D Loss: 1.3991 | G Loss: 0.5537
[Epoch 1/50] [Batch 200/469] D Loss: 1.4043 | G Loss: 0.6413
[Epoch 1/50] [Batch 400/469] D Loss: 1.3963 | G Loss: 0.6932
[Epoch 2/50] [Batch 0/469] D Loss: 1.3850 | G Loss: 0.6516
[Epoch 2/50] [Batch 200/469] D Loss: 1.3892 | G Loss: 0.6514
[Epoch 2/50] [Batch 400/469] D Loss: 1.3865 | G Loss: 0.7049
[Epoch 3/50] [Batch 0/469] D Loss: 1.3842 | G Loss: 0.6760
[Epoch 3/50] [Batch 200/469] D Loss: 1.3836 | G Loss: 0.6859
[Epoch 3/50] [Batch 400/469] D Loss: 1.3827 | G Loss: 0.6929
[Epoch 4/50] [Batch 0/469] D Loss: 1.3832 | G Loss: 0.6872
[Epoch 4/50] [Batch 200/469] D Loss: 1.3891 | G Loss: 0.7016
[Epoch 4/50] [Batch 400/469] D Loss: 1.3916 | G Loss: 0.7188
[Epoch 5/50] [Batch 0/469] D Loss: 1.3720 | G Loss: 0.6894
[Epoch 5/50] [Batch 200/469] D Loss: 1.3890 | G Loss: 0.6892
[Epoch 5/50] [Batch 400/469] D Loss: 1.3866 | G Loss: 0.7187
[Epoch 6/50] [Batch 0/469] D Loss: 1.3746 | G Loss: 0.6853
[Epoch 6/50] [Batch 200/469] D Loss:

In [23]:
def generate_digit_images(generator, digit, num_samples=16, save_path = None):
    generator.eval()
    z = torch.randn(num_samples, z_dim).to(device)
    labels = torch.full((num_samples,), digit, dtype=torch.long).to(device)
    
    with torch.no_grad():
        gen_imgs = generator(z, labels)
        gen_imgs = gen_imgs * 0.5 + 0.5
    
    if save_path:
        save_image(gen_imgs, save_path, nrow=4)
        print(f"Saved to {save_path}")
    return gen_imgs

In [24]:
generate_digit_images(generator, digit=7, num_samples=16, save_path="cgan_generated/seven.png")

Saved to cgan_generated/seven.png


tensor([[[[5.9605e-08, 3.1292e-06, 1.0639e-05,  ..., 5.7220e-06,
           3.0696e-06, 1.4603e-06],
          [1.7881e-07, 2.7418e-06, 7.7486e-07,  ..., 3.8236e-05,
           6.7055e-06, 3.2783e-07],
          [2.6822e-07, 2.2560e-05, 1.1626e-03,  ..., 1.4305e-05,
           5.9605e-08, 2.5332e-06],
          ...,
          [4.7684e-07, 2.0862e-07, 0.0000e+00,  ..., 9.4175e-06,
           6.7186e-04, 3.5763e-07],
          [1.1591e-01, 3.2783e-07, 1.1232e-04,  ..., 4.5002e-06,
           1.2383e-04, 3.0696e-06],
          [2.0862e-07, 3.6657e-06, 5.9605e-08,  ..., 1.3709e-06,
           1.1921e-07, 3.5465e-06]]],


        [[[2.9802e-08, 3.2485e-06, 9.8646e-06,  ..., 4.5002e-06,
           2.5034e-06, 1.5199e-06],
          [1.4901e-07, 2.5332e-06, 6.2585e-07,  ..., 4.7117e-05,
           5.7817e-06, 2.3842e-07],
          [2.6822e-07, 2.3305e-05, 1.2158e-03,  ..., 1.1146e-05,
           5.9605e-08, 2.9504e-06],
          ...,
          [3.5763e-07, 1.7881e-07, 0.0000e+00,  ..., 7.33